# CLONE github repository

In [1]:
!git clone https://github.com/godSaveDaniele/LUPIN-steering.git /kaggle/working/LUPIN
%cd /kaggle/working/LUPIN

Cloning into '/kaggle/working/LUPIN'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (273/273), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 273 (delta 118), reused 241 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (273/273), 18.95 MiB | 12.64 MiB/s, done.
Resolving deltas: 100% (118/118), done.
/kaggle/working/LUPIN


## Eventuale aggiornamento repository

In [2]:
#!git pull --ff-only origin main

# LIBRERIE

In [3]:
%pip install "transformers==4.38.2" "jellyfish==1.0.3" "Jinja2==3.1.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 16.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.5 MB/s eta 0:00:00
  Created wheel for jellyfish: filename=jellyfish-1.0.3-cp313-cp313-linux_x86_64.whl size=348103 sha256=6b7d5050c76d57f71825c50d5fabb337f50753760fab0f3c9bd8a61c64fe8a5e
  Stored in directory: /root/.cache/pip/wheels/d6/5b/63/ea0b4bb5e3552fd47118f6d198b4ef567c1bef7fadecef283e
Successfully built jellyfish
  Attempting uninstall: Jinja2
    Found

# IMPORT + GPU 

In [4]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), "Abilita la GPU nelle impostazioni Kaggle."
print("GPU:", torch.cuda.get_device_name(0))

#Path("log_history/bpic2017_o").mkdir(parents=True, exist_ok=True)

GPU: Tesla T4


# LUPIN fine tuning

In [5]:
#!python -u -m main bpic2017_o

# RANDOM SUBSET fine tuning


In [6]:
#!python -u constrained_fine_tuning.py \
#   --dataset bpic2017_o \
#    --checkpoint /kaggle/input/datasets/danieleolivet/lupin-bpic2017-o/bpic2017_o_all.pth \
#    --template Response \
#    --activation "O_Returned" \
#    --target "O_Accepted" \
#    --mode heads_only \
#    --learning-rate 1e-5 \
#    --epochs 6 \
#    --batch-size 16 \
#    --seed 42 \
#    --random-subset

# SEMANTIC FINE TUNING

In [7]:
#!python -u sematic_fine_tuning.py \
#    --dataset bpic2017_o \
#    --checkpoint /kaggle/input/datasets/danieleoliveti19/lupin-bpic-2017-o/bpic2017_o_all.pth \
#    --activation "O_Sent (mail and online)" \
#    --target O_Returned \
#    --mode heads_only \
#    --epochs 10 \
#    --batch-size 16 \
#    --learning-rate 1e-5 \
#    --seed 42 \
#    --dual-objective \
#    --lambda 1

# Evaluation LUPIN

prima di valutare il modello pre-trained e necessario effettuare il pre-processing per generare il file di test

In [8]:
#from pathlib import Path
#from preprocessing.log_to_history import Log

#Path("log_history/bpic2017_o").mkdir(parents=True, exist_ok=True)

#Log("bpic2017_o", "all") #pre-processing 

In [9]:
#!python -u eval_model.py bpic2017_o \
#    --checkpoint /kaggle/input/datasets/danieleolivet/lupin-bpic2017-o/bpic2017_o_all.pth \
#    --template Response \
#    --activation "O_Returned" \
#    --target "O_Accepted" 

# Evaluation CONSTRAINED LUPIN

In [10]:
#!python -u eval_model.py bpic2017_o \
#    --checkpoint /kaggle/working/LUPIN/models/bpic2017_o/constrained/best.pth \
#    --template Response \
#    --activation "O_Returned" \
#    --target "O_Accepted" 


# STEERED MODEL Evaluation

In [11]:
!python -u weight_steering.py \
    --dataset bpic2017_o \
    --base-checkpoint /kaggle/input/datasets/danieleoliveti19/lupin-bpic-2017-o/bpic2017_o_all.pth \
    --constrained-checkpoint /kaggle/input/datasets/danieleoliveti19/semantic-finetuning-vincolo2/best.pth \
    --template Response \
    --activation "O_Sent (mail and online)" \
    --target "O_Returned"  \
    --alphas 0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1 1.1 1.2 1.5


Tensors modified trough fine tuning:
output_layers.0.weight | shape=(9, 512) | not null components=4608/4608 | norm=1.89974
output_layers.0.bias | shape=(9,) | not null components=9/9 | norm=0.0447107
output_layers.1.weight | shape=(9, 512) | not null components=4608/4608 | norm=0.709144
output_layers.1.bias | shape=(9,) | not null components=9/9 | norm=0.040332
output_layers.2.weight | shape=(9, 512) | not null components=4608/4608 | norm=0.644467
output_layers.2.bias | shape=(9,) | not null components=9/9 | norm=0.0473004
output_layers.3.weight | shape=(9, 512) | not null components=4608/4608 | norm=0.566819
output_layers.3.bias | shape=(9,) | not null components=9/9 | norm=0.0592705
output_layers.4.weight | shape=(9, 512) | not null components=4608/4608 | norm=0.587413
output_layers.4.bias | shape=(9,) | not null components=9/9 | norm=0.0637994

===== ALPHA = 0.0 =====
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is 

In [12]:
!python -u combined_steering.py \
    --dataset bpic2017_o \
    --base-checkpoint /kaggle/input/datasets/danieleoliveti19/lupin-bpic-2017-o/bpic2017_o_all.pth \
    --checkpoint-1 /kaggle/input/datasets/danieleoliveti19/semantic-loss-fine-tuning/best.pth \
    --checkpoint-2 /kaggle/input/datasets/danieleoliveti19/semantic-finetuning-vincolo2/best.pth \
    --template-1 Response \
    --activation-1 O_Returned \
    --target-1 O_Accepted \
    --template-2 Response \
    --activation-2 "O_Sent (mail and online)" \
    --target-2 O_Returned \
    --alphas 0 0.2 0.4 0.6 0.8 1.0
    --betas 0 0.2 0.4 0.6 0.8 1.0 \
    --batch-size 16

IndentationError: unexpected indent (1884281098.py, line 2)